In [5]:
import os
import sqlite3
import pandas as pd

csv_path = 'dataset.csv/dataset.csv'

try:
    if not os.path.exists(csv_path):
        raise FileNotFoundError(f"No se encontró el archivo: {csv_path}")

    df = pd.read_csv(csv_path)
    print(f"Dataset cargado: {df.shape[0]} filas, {df.shape[1]} columnas")
    print("Columnas disponibles:", list(df.columns))

    conn = sqlite3.connect('spotify.db')
    cursor = conn.cursor()
    cursor.execute("PRAGMA foreign_keys = OFF;")

    # 1. albums
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS albums (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            album_name TEXT,
            artists TEXT,
            popularity INTEGER
        )
    ''')

    cols_albums = ['track_id', 'track_name', 'album_name', 'artists', 'popularity']
    df_albums = df[cols_albums].drop_duplicates(subset=['track_id']).fillna('')
    tuplas_albums = [tuple(row) for row in df_albums.itertuples(index=False, name=None)]

    cursor.executemany('''
        INSERT OR REPLACE INTO albums
        VALUES (?, ?, ?, ?, ?)
    ''', tuplas_albums)

    print(f"✅ Insertados {len(tuplas_albums)} álbumes únicos")

    # 2. genre
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS genre (
            track_id TEXT PRIMARY KEY,
            track_name TEXT NOT NULL,
            track_genre TEXT
        )
    ''')

    cols_genre = ['track_id', 'track_name', 'track_genre']
    df_genre = df[cols_genre].drop_duplicates(subset=['track_id']).fillna('')
    tuplas_genre = [tuple(row) for row in df_genre.itertuples(index=False, name=None)]

    cursor.executemany('''
        INSERT OR REPLACE INTO genre
        VALUES (?, ?, ?)
    ''', tuplas_genre)

    print(f"✅ Insertados {len(tuplas_genre)} géneros únicos")

    # 3. Audio_Features
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Audio_Features (
            track_id TEXT PRIMARY KEY,
            danceability REAL NOT NULL,
            energy REAL NOT NULL,
            "key" INTEGER,
            loudness REAL,
            mode INTEGER,
            speechiness REAL,
            acousticness REAL,
            instrumentalness REAL,
            liveness REAL,
            valence REAL,
            tempo REAL,
            time_signature INTEGER
        )
    ''')

    cols_features = [
        'track_id', 'danceability', 'energy', 'key', 'loudness', 'mode',
        'speechiness', 'acousticness', 'instrumentalness', 'liveness',
        'valence', 'tempo', 'time_signature'
    ]

    df_features = df[cols_features].drop_duplicates(subset=['track_id']).fillna(0)
    tuplas_features = [tuple(row) for row in df_features.itertuples(index=False, name=None)]

    cursor.executemany('''
        INSERT OR REPLACE INTO Audio_Features
        VALUES (?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?, ?)
    ''', tuplas_features)

    print(f"✅ Insertados {len(tuplas_features)} features únicos")

    # 4. Audio_Analysis
    cursor.execute('''
        CREATE TABLE IF NOT EXISTS Audio_Analysis (
            track_id TEXT PRIMARY KEY,
            dance_energy_sum REAL NOT NULL,
            dance_energy_avg REAL NOT NULL,
            is_dance_track INTEGER,
            is_energy_high INTEGER,
            tempo_category TEXT,
            total_features_score REAL
        )
    ''')

    cursor.execute('DELETE FROM Audio_Analysis')

    cursor.execute('''
        INSERT INTO Audio_Analysis (
            track_id,
            dance_energy_sum,
            dance_energy_avg,
            is_dance_track,
            is_energy_high,
            tempo_category,
            total_features_score
        )
        SELECT
            track_id,
            (danceability + energy) AS dance_energy_sum,
            ((danceability + energy) / 2) AS dance_energy_avg,
            CASE WHEN (danceability + energy) > 1.2 THEN 1 ELSE 0 END AS is_dance_track,
            CASE WHEN energy > 0.7 THEN 1 ELSE 0 END AS is_energy_high,
            CASE
                WHEN tempo < 90 THEN 'Lento'
                WHEN tempo < 120 THEN 'Medio'
                ELSE 'Rapido'
            END AS tempo_category,
            (danceability + energy + valence) AS total_features_score
        FROM Audio_Features
    ''')

    conn.commit()
    conn.close()

    print("🎉 Base de datos creada correctamente: spotify.db")

except FileNotFoundError as e:
    print(f"❌ {e}")

except KeyError as e:
    print(f"❌ Falta una columna en el CSV: {e}")

except Exception as e:
    print(f"❌ Error general: {e}")

❌ No se encontró el archivo: dataset.csv/dataset.csv
